# BL importer finalization & debug

## Imports

In [1]:
import os
import json

from impresso_essentials.utils import ALL_MEDIA, PARTNER_TO_MEDIA
from copy import deepcopy
from tqdm import tqdm
import pandas as pd
import bs4
from random import shuffle
from text_preparation.importers.bl.detect import BlIssueDir, dir2issue, detect_issues, select_issues
from text_preparation.importers.bl.omni.classes import BlOmniNewspaperPage, BlOmniNewspaperIssue
from PIL import Image, ImageDraw, ImageFont
from text_preparation.utils import draw_box_on_img, coords_to_xywh, coords_to_xy, rescale_coords
from text_preparation.importers.mets_alto.alto import distill_coordinates
from text_preparation.importers import (
    CONTENTITEM_TYPES,
    CONTENTITEM_TYPE_IMAGE,
    CONTENTITEM_TYPE_ADVERTISEMENT,
)
IIIF_ENDPOINT_URI = "https://impresso-project.ch/api/proxy/iiif/"
IIIF_SUFFIX = "info.json"

## OmniPage Format

First, adapt BL_ocr_formats.json file to only keep the Aliases and issues corresponding to OmniPage-NLP format.
That will allow to only detect titles for this format and will already help a lot.
The document could have all the issues or be much much smaller with only Alias > NLP > list of dates for the given format. Or directly have the list of paths to mets files of the correct format.

Then the BL_extended_title_list.csv should also be processed to go from alias > NLP > date range > working title and variant titles so that each issue can have its variant title attached to it.

In [2]:
bl_source_data_dir = "/mnt/project_impresso/original/BL"
bl_w_source_data_dir = "/mnt/impresso_ocr_BL"
bl_sample_dir = "/home/piconti/impresso-text-acquisition/text_preparation/data/sample_data/BL"
bl_formats_filename = "BL_ocr_formats.json"
bl_titles_filename = "BL_extended_title_list.csv"
alias_to_NLP_filename = "BL_alias_to_NLP.json"
bl_titles_out_filename = "BL_all_titles.json"
bl_alias_25_08_formats = "/home/piconti/impresso-text-acquisition/text_preparation/data/sample_data/BL/BL_ocr_formats_bl_alias_2025-08-25.json"
bl_media_list_ext_path = os.path.join(bl_sample_dir, bl_titles_filename)
alias_to_NLP = os.path.join(bl_sample_dir, alias_to_NLP_filename)
bl_format_specific_issues = "BL_{format}_issues.json"
RENAMING_INFO_FILE = "renaming_info.json"
BL_IMG_TYPE = "illustration"
BL_AD_TYPE = "advert"
BL_CAPTION_TYPE = "caption"

ocr_formats = ["OmniPage-NLP", "BL-ALIAS", "Nuance-NLP", "ABBYY-ALIAS", "ABBYY-NLP"]

In [3]:
with open(os.path.join(bl_source_data_dir, bl_formats_filename), "r", encoding='utf-8') as fin:
    bl_ocr_formats = json.load(fin)

print(f"Reading: {os.path.join(bl_source_data_dir, bl_formats_filename)}")

print(f"There are {len(bl_ocr_formats)} aliases in {bl_formats_filename}")

Reading: /mnt/project_impresso/original/BL/BL_ocr_formats.json
There are 368 aliases in BL_ocr_formats.json


Some preprocessing steps were necessary to go from the `BL_ocr_formats.json` file to format-specific issue lists. 
In addition, a list of variant titles for each alias was also prepared.

All this is in the `bl_omni_importer_prep.ipynb` jupyter notebook. 
This notebook is a continuation of it, starting directly with the detect/select functions, and then the NewspaperIssue and NewspaperPage classes

### 3. Detect/Select functions

In [ ]:
omni_issues = mets_paths_per_format["OmniPage-NLP"]
omni_issues

In [ ]:
all_issues = []
for alias, issues_of_alias in omni_issues.items():
    
    issue_paths = [dir2issue(path) for path in list(issues_of_alias['priority_issues'].keys())]
    print(f"{alias} - Found {len(issue_paths)} issues")
    all_issues.extend(issue_paths)

all_issues[10:-10]

In [ ]:
config_test_1 = {
    "titles": {
        "ILOL": [],
        "AGMO": [],
        "AATA": [],
        "AGE52": [],
        "AHEC": [],
        "ANJO": [],
        "BHCH": [],
        "BHFA": []
    },
    "exclude_titles": [],
    "year_only": False
}

detected = detect_issues(bl_source_data_dir)

print(f"Detected {len(detected)} issues in total")

for title in config_test_1['titles']:
    print(f"Detected {len([i for i in detected if i.alias == title])} issues for {title}")

Detected 161486 issues in total
Detected 23 issues for ILOL
Detected 691 issues for AGMO
Detected 31 issues for AATA
Detected 37 issues for AGE52
Detected 208 issues for AHEC
Detected 0 issues for ANJO
Detected 0 issues for BHCH


In [5]:
selected = select_issues(bl_source_data_dir, config = config_test_1)

print(f"Selected  {len(selected)} issues in total")

for title in config_test_1['titles']:
    print(f"Detected {len([i for i in selected if i.alias == title])} issues for {title}")

Selected  990 issues in total
Detected 23 issues for ILOL
Detected 691 issues for AGMO
Detected 31 issues for AATA
Detected 37 issues for AGE52
Detected 208 issues for AHEC
Detected 0 issues for ANJO
Detected 0 issues for BHCH


In [ ]:
config_test_2 = {
    "titles": {
        "AGMO": "1817/08/03-1817/12/21",
        "AATA": "1848/01/01-1849/01/01",
        "AGE52": [],
        "AHEC": ["1880/02/21", "1877/02/10"],
        "ANJO": [],
        "BHCH": [],
    },
    "exclude_titles": [],
    "year_only": False
}

selected = select_issues(bl_source_data_dir, config = config_test_2)

print(f"Selected  {len(selected )} issues in total")

for title in config_test_2['titles']:
    print(f"Detected {len([i for i in selected if i.alias == title])} issues for {title}")

Selected  60 issues in total
Detected 21 issues for AGMO
Detected 0 issues for AATA
Detected 37 issues for AGE52
Detected 2 issues for AHEC
Detected 0 issues for ANJO
Detected 0 issues for BHCH


### 4. BlOmniNewspaperIssue Class

#### Debug issue

In [47]:
config_test_1 = {
    "titles": {
        "ILOL": [],
        "AGMO": [],
        "AATA": [],
        "AGE52": [],
        "AHEC": [],
        "ANJO": [],
        "BHCH": [],
    },
    "exclude_titles": [],
    "year_only": False
}

detected = detect_issues(bl_source_data_dir)

print(f"Detected {len(detected)} issues in total")

selected = select_issues(bl_source_data_dir, config = config_test_1)

print(f"Selected  {len(selected)} issues in total")

for title in config_test_1['titles']:
    print(f"Detected {len([i for i in selected if i.alias == title])} issues for {title}")

Detected 161486 issues in total
Selected  990 issues in total
Detected 23 issues for ILOL
Detected 691 issues for AGMO
Detected 31 issues for AATA
Detected 37 issues for AGE52
Detected 208 issues for AHEC
Detected 0 issues for ANJO
Detected 0 issues for BHCH


In [51]:
test_issue = selected[0]
test_issue

IssueDirectory(provider='BL', alias='ILOL', date=datetime.date(1843, 5, 21), edition='a', path='/mnt/project_impresso/original/BL/ILOL/0003005/1843/05/21', nlp='0003005')

In [52]:
os.listdir(test_issue.path)

['renaming_info.json',
 '0003005_18430521_0007.xml',
 '0003005_18430521_0009.xml',
 '0003005_18430521_0011.xml',
 '0003005_18430521_0008.xml',
 '0003005_18430521_0006.xml',
 '0003005_18430521_0012.xml',
 '0003005_18430521_0004.xml',
 '0003005_18430521_mets.xml',
 '0003005_18430521_0003.xml',
 '0003005_18430521_0002.xml',
 '0003005_18430521_0005.xml',
 '0003005_18430521_0010.xml',
 '0003005_18430521_0001.xml']

In [53]:
with open(os.path.join(test_issue.path, RENAMING_INFO_FILE), 'r') as fin:
    test_issue_renaming_info = json.load(fin)

test_issue_renaming_info

{'12': {'original_filename': '0003005_18430521_0012.jp2',
  'new_filename': 'ILOL-1843-05-21-a-p0012.jp2',
  'issue_id': 'ILOL-1843-05-21-a',
  'original_nlp': '0003005',
  'img_dir_path': '/mnt/impresso_images_BL/ILOL/1843/05/21/a',
  'ocr_dir_path': '/mnt/project_impresso/original/BL/ILOL/0003005/1843/05/21',
  'width': 4409,
  'height': 6504},
 '1': {'original_filename': '0003005_18430521_0001.jp2',
  'new_filename': 'ILOL-1843-05-21-a-p0001.jp2',
  'issue_id': 'ILOL-1843-05-21-a',
  'original_nlp': '0003005',
  'img_dir_path': '/mnt/impresso_images_BL/ILOL/1843/05/21/a',
  'ocr_dir_path': '/mnt/project_impresso/original/BL/ILOL/0003005/1843/05/21',
  'width': 4409,
  'height': 6504},
 '5': {'original_filename': '0003005_18430521_0005.jp2',
  'new_filename': 'ILOL-1843-05-21-a-p0005.jp2',
  'issue_id': 'ILOL-1843-05-21-a',
  'original_nlp': '0003005',
  'img_dir_path': '/mnt/impresso_images_BL/ILOL/1843/05/21/a',
  'ocr_dir_path': '/mnt/project_impresso/original/BL/ILOL/0003005/1843

In [54]:
bl_issue = BlOmniNewspaperIssue(test_issue)
bl_issue

In [55]:
bl_issue.issue_data

{'id': 'ILOL-1843-05-21-a',
 'cdt': '2025-09-01 16:38:26',
 'ts': '2025-09-01T14:38:26Z',
 'st': 'newspaper',
 'sm': 'print',
 'olr': True,
 'i': [{'m': {'id': 'ILOL-1843-05-21-a-i0001',
    'tp': 'article',
    'pp': [1],
    'var_t': 'Illustrated London Life',
    'lg': 'en',
    'ro': 1},
   'l': {'bl_nlp': '0003005',
    'src_files': {'mets_xml': '0003005_18430521_mets.xml',
     'alto_xml': ['0003005_18430521_0001.xml'],
     'page_image': ['0003005_18430521_0001.jp2']},
    'id': 'art0001',
    'parts': [{'comp_role': 'pagearea',
      'comp_id': 'pa0001001',
      'comp_label': 'textblock',
      'comp_fileid': 'img0001-alto',
      'comp_page_no': 1},
     {'comp_role': 'pagearea',
      'comp_id': 'pa0001002',
      'comp_label': 'textblock',
      'comp_fileid': 'img0001-alto',
      'comp_page_no': 1},
     {'comp_role': 'pagearea',
      'comp_id': 'pa0001003',
      'comp_label': 'textblock',
      'comp_fileid': 'img0001-alto',
      'comp_page_no': 1},
     {'comp_role':

##### Adapt `_parse_content_item_logic()` for handle images and store the original filename

In [56]:
mets_doc = bl_issue.xml
content_items = []

# Get logical structure of issue
divs = (
    mets_doc.find("structMap", {"TYPE": "LOGICAL"})
    .find("div", {"TYPE": "ISSUE"})
    .findChildren("div")
)

# Sort to have same naming
#sorted_divs = sorted(divs, key=lambda x: int(''.join(filter(str.isdigit, x.get("DMDID")))))

#sorted_divs
divs

[<mets:div DMDID="modsarticle1" ID="art0001" TYPE="ARTICLE"/>,
 <mets:div DMDID="modsarticle2" ID="art0002" TYPE="ARTICLE"/>,
 <mets:div DMDID="modsarticle3" ID="art0003" TYPE="ARTICLE"/>,
 <mets:div DMDID="modsarticle4" ID="art0004" TYPE="ARTICLE"/>,
 <mets:div DMDID="modsarticle5" ID="art0005" TYPE="ARTICLE"/>,
 <mets:div DMDID="modsarticle6" ID="art0006" TYPE="ARTICLE"/>,
 <mets:div DMDID="modsarticle7" ID="art0007" TYPE="ARTICLE"/>,
 <mets:div DMDID="modsarticle8" ID="art0008" TYPE="ARTICLE"/>,
 <mets:div DMDID="modsarticle9" ID="art0009" TYPE="ARTICLE"/>,
 <mets:div DMDID="modsarticle10" ID="art0010" TYPE="ARTICLE"/>,
 <mets:div DMDID="modsarticle11" ID="art0011" TYPE="ARTICLE"/>,
 <mets:div DMDID="modsarticle12" ID="art0012" TYPE="ARTICLE"/>,
 <mets:div DMDID="modsarticle13" ID="art0013" TYPE="ARTICLE"/>,
 <mets:div DMDID="modsarticle14" ID="art0014" TYPE="ARTICLE"/>,
 <mets:div DMDID="modsarticle15" ID="art0015" TYPE="ARTICLE"/>,
 <mets:div DMDID="modsarticle16" ID="art0016" TYP

In [17]:
found_types = set(x.get("TYPE") for x in divs)

phys_structmap = mets_doc.find("structMap", {"TYPE": "PHYSICAL"})
structlink = mets_doc.find("structLink")

print(found_types)
print(phys_structmap)
print(structlink)

{'ADVERT', 'ARTICLE'}
<mets:structMap LABEL="Physical Structure" TYPE="PHYSICAL">
<mets:div DMDID="MODS_ISSUE_0003005-00000" ID="phys0" LABEL="Illustrated London Life. 1843-05-21" TYPE="physSequence">
<mets:div ID="phys1" ORDER="1" ORDERLABEL="1" TYPE="page">
<mets:fptr FILEID="img0001-master"/>
<mets:fptr FILEID="img0001-alto"/>
<mets:div ID="pa0001001" LABEL="Textblock" TYPE="pagearea">
<mets:fptr>
<mets:area COORDS="228,3665,1401,4533" FILEID="img0001-master" SHAPE="RECT"/>
</mets:fptr>
<mets:fptr>
<mets:area BEGIN="word002399" BETYPE="IDREF" END="word002574" FILEID="img0001-alto"/>
</mets:fptr>
</mets:div>
<mets:div ID="pa0001002" LABEL="Textblock" TYPE="pagearea">
<mets:fptr>
<mets:area COORDS="1428,3666,2605,4167" FILEID="img0001-master" SHAPE="RECT"/>
</mets:fptr>
<mets:fptr>
<mets:area BEGIN="word002575" BETYPE="IDREF" END="word002667" FILEID="img0001-alto"/>
</mets:fptr>
</mets:div>
<mets:div ID="pa0001003" LABEL="Textblock" TYPE="pagearea">
<mets:fptr>
<mets:area COORDS="1431

##### `_parse_content_item`

In [15]:
def _get_part_dict(div, comp_role: str | None):
    """Construct the parts for a certain div entry of METS.

    Args:
        div (Tag): Content item div
        comp_role (str | None): Role of the component

    Returns:
        dict[str, Any]: Parts dict for given div.
    """
    comp_fileid = div.find("area", {"BETYPE": "IDREF"}).get("FILEID")
    comp_id = div.get("ID")
    comp_page_no = int(div.parent.get("ORDER"))
    # This is where illustrations will be identified
    comp_label = div.get("LABEL").lower()
    if comp_role is None:
        type_attr = div.get("TYPE")
        comp_role = type_attr.lower() if type_attr else None

    return {
        "comp_role": comp_role,
        "comp_id": comp_id,
        "comp_label": comp_label,
        "comp_fileid": comp_fileid,
        "comp_page_no": int(comp_page_no),
    }

In [38]:
def _get_image_and_captions(div, part_id, div_parts, curr_ci_parts, ci_image_parts, last_img_part_id):
    # for each illustration, store its coordinates and any potential caption
    if div.get("LABEL").lower() == BL_IMG_TYPE:
        img_xy_coords = div.find("area", {"SHAPE": "RECT"}).get("COORDS")
        # directly convert the coordinates to the wanted xywh format
        div_parts["coords"] = coords_to_xywh([int(c) for c in img_xy_coords.split(',')])
        if part_id not in ci_image_parts:
            ci_image_parts[part_id] = [div_parts]
        else:
            ci_image_parts[part_id].append(div_parts)
        # keep track of which illustration it is to make sure we can connect them back after
        last_img_part_id = part_id

    # if the next element is a caption, attach it directly
    if div.get("LABEL").lower() == BL_CAPTION_TYPE:
        if curr_ci_parts[-1]['comp_id'] == last_img_part_id:
            #ci_image_parts[last_img_part_id]["caption_parts"] = div_parts
            cap_xy_coords = div.find("area", {"SHAPE": "RECT"}).get("COORDS")
            # directly convert the coordinates to the wanted xywh format
            div_parts["coords"] = coords_to_xywh([int(c) for c in cap_xy_coords.split(',')])
            ci_image_parts[last_img_part_id].append(div_parts)
            #ci_image_parts[last_img_part_id]["caption_coords"] = coords_to_xywh([int(c) for c in cap_xy_coords.split(',')])
        else:
            print(f"curr_ci_parts[-1]: {curr_ci_parts[-1]}, last_img_part_id={last_img_part_id}")
            msg = f"{bl_issue.id}, {div_parts['comp_page_no']} - caption {div.get('ID')} does not follow an illustration!"
            print(msg)
            bl_issue._notes.append(msg)

    return ci_image_parts, last_img_part_id

In [60]:
counter = 2

item_div = divs[1]
#for div in divs:
#only for first for now
    # Parse Each contentitem
item_dmd_sec = mets_doc.find("dmdSec", {"ID": item_div.get("DMDID")})
    #content_items.append(test_issue._parse_content_item(div, counter, phys_structmap, structlink, dmd_sec))
 
div_type = item_div.get("TYPE").lower()

div_id = item_div.get("ID")

lang = item_dmd_sec.findChild("languageTerm").text
title = item_dmd_sec.findChild("title").text

div_id, item_dmd_sec, div_type, lang, title

('art0002',
 <mets:dmdSec ID="modsarticle2">
 <mets:mdWrap MDTYPE="MODS">
 <mets:xmlData>
 <mods:mods>
 <mods:titleInfo ID="modsarticle2_TI1" xml:lang="en">
 <mods:title>LORD CARDIGAN IN DUBLIN.</mods:title>
 </mods:titleInfo>
 <mods:language>
 <mods:languageTerm authority="rfc3066" type="code">en</mods:languageTerm>
 </mods:language>
 </mods:mods>
 </mets:xmlData>
 </mets:mdWrap>
 </mets:dmdSec>,
 'article',
 'en',
 'LORD CARDIGAN IN DUBLIN.')

In [ ]:
counter = 1
content_items = []
cis_img_parts = []

for idx, div in enumerate(divs):
    print(f"\n---- div #{idx} -----")
    print(f"counter = {counter}")
    dmd_sec = mets_doc.find("dmdSec", {"ID": div.get("DMDID")})
    div_type = div.get("TYPE").lower()

    item_dmd_sec = mets_doc.find("dmdSec", {"ID": div.get("DMDID")})
    lang = item_dmd_sec.findChild("languageTerm")

    if div_type == BL_IMG_TYPE:
        div_type = CONTENTITEM_TYPE_IMAGE
    elif div_type == BL_AD_TYPE:
        div_type = CONTENTITEM_TYPE_ADVERTISEMENT
    
    tag = tag = f"#{div.get('ID')}"
    print(tag)
    linkgrp = structlink.find("smLocatorLink", {"xlink:href": tag}).parent

    # Remove `#` from xlink:href
    div_parts_ids = [
        x.get("xlink:href")[1:]
        for x in linkgrp.findAll("smLocatorLink")
        if x.get("xlink:href") != tag
    ]

    ci_parts = []
    ci_image_parts = {}
    last_img_part_id = None
    last_img_part_idx = None
    for idx, p in enumerate(div_parts_ids):
        # Get element in physical map
        part_div = phys_structmap.find("div", {"ID": p})
        print(f"div {idx} from div parts: {div}")
        type_attr = part_div.get("TYPE")
        comp_role = type_attr.lower() if type_attr else None

        # In that case, need to add all parts
        if comp_role == "page":
            for sub_div in part_div.findAll("div"):
                subdiv_part_dict = _get_part_dict(sub_div, None)
                subdiv_part_id = sub_div.get("ID")
                assert subdiv_part_id == subdiv_part_dict['comp_id']#:
                #print(f"!!!!!!!subdiv_part_id={subdiv_part_id}, subdiv_part_dict['comp_id']:{subdiv_part_dict['comp_id']}")
                
                ci_image_parts, last_img_part_id = _get_image_and_captions(sub_div, subdiv_part_id, subdiv_part_dict, ci_parts, ci_image_parts, last_img_part_id)
                ci_parts.append(subdiv_part_dict)
                
        else:
            div_part_dict = _get_part_dict(part_div, comp_role)
            ci_image_parts, last_img_part_id = _get_image_and_captions(part_div, p, div_part_dict, ci_parts, ci_image_parts, last_img_part_id)
            ci_parts.append(div_part_dict)

    
    content_item = {
        "m": {
            "id": f"{bl_issue.id}-i{str(counter).zfill(4)}",
            "tp": div_type,
            "pp": [],
        },
        "l": {
            "bl_nlp": bl_issue.nlp,
            "id": div.get("ID"),
            "parts": ci_parts,
        },
    }

    if lang is not None:
        content_item['m']["lg"] = lang.text
    for p in content_item["l"]["parts"]:
        pge_no = p["comp_page_no"]
        if pge_no not in content_item["m"]["pp"]:
            content_item["m"]["pp"].append(pge_no)


    content_items.append(content_item)
    cis_img_parts.append(ci_image_parts)
    counter += 1

In [40]:
content_items[43]

{'m': {'id': 'ILOL-1843-05-21-a-i0044',
  'tp': 'article',
  'pp': [11],
  'lg': 'en'},
 'l': {'bl_nlp': '0003005',
  'id': 'art0044',
  'parts': [{'comp_role': 'pagearea',
    'comp_id': 'pa0011003',
    'comp_label': 'headline',
    'comp_fileid': 'img0011-alto',
    'comp_page_no': 11},
   {'comp_role': 'pagearea',
    'comp_id': 'pa0011004',
    'comp_label': 'textblock',
    'comp_fileid': 'img0011-alto',
    'comp_page_no': 11},
   {'comp_role': 'pagearea',
    'comp_id': 'pa0011005',
    'comp_label': 'textblock',
    'comp_fileid': 'img0011-alto',
    'comp_page_no': 11},
   {'comp_role': 'pagearea',
    'comp_id': 'pa0011006',
    'comp_label': 'textblock',
    'comp_fileid': 'img0011-alto',
    'comp_page_no': 11},
   {'comp_role': 'pagearea',
    'comp_id': 'pa0011007',
    'comp_label': 'textblock',
    'comp_fileid': 'img0011-alto',
    'comp_page_no': 11},
   {'comp_role': 'pagearea',
    'comp_id': 'pa0011008',
    'comp_label': 'textblock',
    'comp_fileid': 'img0011-a

In [41]:
cis_img_parts[43]#38

{'pa0011023': [{'comp_role': 'pagearea',
   'comp_id': 'pa0011023',
   'comp_label': 'illustration',
   'comp_fileid': 'img0011-alto',
   'comp_page_no': 11,
   'coords': [2887, 2183, 789, 575]},
  {'comp_role': 'pagearea',
   'comp_id': 'pa0011024',
   'comp_label': 'caption',
   'comp_fileid': 'img0011-alto',
   'comp_page_no': 11,
   'coords': [3047, 2805, 525, 38]}]}

In [26]:
bl_issue.pages[9]

In [21]:
issue_images_path = "/mnt/impresso_images_BL/ILOL/1843/05/21/a"
pg_filename = 'ILOL-1843-05-21-a-p{num}.jp2'
pg_path = os.path.join(issue_images_path, pg_filename)

In [ ]:
ci_idx = 0
ci = content_items[ci_idx]
img_part = cis_img_parts[ci_idx]
page_n = ci['m']['pp'][0]
pg_1_path = pg_path.format(num=str(page_n).zfill(4))

pg_1_img = Image.open(pg_1_path)
for div_id, parts in img_part.items():
    coords_img_xy = coords_to_xy(parts['coords'])
    print(f"coords_img_xy:{coords_img_xy}")
    pg_1_img = draw_box_on_img(pg_1_path, coords_img_xy, pg_1_img, width=15)
    if 'caption_coords' in parts:
        coords_cap_xy = coords_to_xy(parts['caption_coords'])
        print(f"coords_cap_xy:{coords_cap_xy}")
        #coords_cap_xy = coords_to_xy([int(c) for c in str_coords])
        pg_1_img = draw_box_on_img(pg_1_path, coords_cap_xy, pg_1_img, width=10)
    print(f"showing with {div_id}")
    pg_1_img.show()

In [ ]:
ci_idx = 43
ci = content_items[ci_idx]
img_part = cis_img_parts[ci_idx]
page_n = ci['m']['pp'][0]
pg_1_path = pg_path.format(num=str(page_n).zfill(4))

pg_1_img = Image.open(pg_1_path)
for div_id, parts in img_part.items():
    coords_img_xy = [int(c) for c in parts['coords'].split(',')]
    #coords_img_xy = coords_to_xy([int(c) for c in str_coords])
    coords_img_xywh = coords_to_xywh(coords_img_xy)
    print(f"coords_img_xy:{coords_img_xy}, coords_img_xywh:{coords_img_xywh}")
    pg_1_img = draw_box_on_img(pg_1_path, coords_img_xy, pg_1_img, width=15)
    if 'caption_coords' in parts:
        coords_cap_xy = [int(c) for c in parts['caption_coords'].split(',')]
        #coords_cap_xy = coords_to_xy([int(c) for c in str_coords])
        coords_cap_xywh = coords_to_xywh(coords_cap_xy)
        print(f"coords_cap_xy:{coords_cap_xy}, coords_cap_xywh:{coords_cap_xywh}")
        pg_1_img = draw_box_on_img(pg_1_path, coords_cap_xy, pg_1_img, width=10)
    print(f"showing with {div_id}")
    pg_1_img.show()

In [ ]:
problem_div = divs[37]
problem_div

tag = tag = f"#{problem_div.get('ID')}"
linkgrp = structlink.find("smLocatorLink", {"xlink:href": tag}).parent
print(tag, linkgrp)

pb_div_parts_ids = [
    x.get("xlink:href")[1:]
    for x in linkgrp.findAll("smLocatorLink")
    if x.get("xlink:href") != tag
]

print(f"pb_div_parts = {pb_div_parts_ids}")

prb_div_part_div = phys_structmap.find("div", {"ID": pb_div_parts_ids[0]})

comp_role = prb_div_part_div.get('TYPE').lower() if type_attr else None

print(f"comp_role = {comp_role}")

prb_div_part_div

##### `_parse_content_parts`

In [ ]:
tag = tag = f"#{item_div.get('ID')}"
print(tag)

linkgrp = structlink.find("smLocatorLink", {"xlink:href": tag}).parent

# Remove `#` from xlink:href
parts_ids = [
    x.get("xlink:href")[1:]
    for x in linkgrp.findAll("smLocatorLink")
    if x.get("xlink:href") != tag
]

parts_ids

#art0001
<mets:smLinkGrp>
<mets:smLocatorLink xlink:href="#art0001" xlink:label="article" xlink:type="locator"/>
<mets:smLocatorLink xlink:href="#pa0001001" xlink:label="page1 area1" xlink:type="locator"/>
<mets:smLocatorLink xlink:href="#pa0001002" xlink:label="page1 area2" xlink:type="locator"/>
<mets:smLocatorLink xlink:href="#pa0001003" xlink:label="page1 area3" xlink:type="locator"/>
<mets:smLocatorLink xlink:href="#pa0001004" xlink:label="page1 area4" xlink:type="locator"/>
<mets:smLocatorLink xlink:href="#pa0001005" xlink:label="page1 area5" xlink:type="locator"/>
<mets:smLocatorLink xlink:href="#pa0001006" xlink:label="page1 area6" xlink:type="locator"/>
<mets:smLocatorLink xlink:href="#pa0001007" xlink:label="page1 area7" xlink:type="locator"/>
<mets:smArcLink ARCTYPE="logicalphysical" xlink:from="article" xlink:to="page1 area1" xlink:type="arc"/>
<mets:smArcLink ARCTYPE="logicalphysical" xlink:from="article" xlink:to="page1 area2" xlink:type="arc"/>
<mets:smArcLink ARCTYPE="l

['pa0001001',
 'pa0001002',
 'pa0001003',
 'pa0001004',
 'pa0001005',
 'pa0001006',
 'pa0001007']

In [15]:
def _get_part_dict(div, comp_role: str | None):
    """Construct the parts for a certain div entry of METS.

    Args:
        div (Tag): Content item div
        comp_role (str | None): Role of the component

    Returns:
        dict[str, Any]: Parts dict for given div.
    """
    comp_fileid = div.find("area", {"BETYPE": "IDREF"}).get("FILEID")
    comp_id = div.get("ID")
    comp_page_no = int(div.parent.get("ORDER"))
    # This is where illustrations will be identified
    comp_label = div.get("LABEL").lower()
    if comp_role is None:
        type_attr = div.get("TYPE")
        comp_role = type_attr.lower() if type_attr else None

    return {
        "comp_role": comp_role,
        "comp_id": comp_id,
        "comp_label": comp_label,
        "comp_fileid": comp_fileid,
        "comp_page_no": int(comp_page_no),
    }

In [17]:
div = phys_structmap.find("div", {"ID": parts_ids[0]})

div.get("LABEL")

'Textblock'

In [24]:
found_divs = phys_structmap.find_all('div', {"ID": lambda x: x in parts_ids})

all(d.get('LABEL') for d in found_divs)

True

In [16]:
parts = []
image_parts = {}
last_img_part_id = None
last_img_part_idx = None
for idx, p in enumerate(parts_ids):
    # Get element in physical map
    div = phys_structmap.find("div", {"ID": p})
    type_attr = div.get("TYPE")
    comp_role = type_attr.lower() if type_attr else None

    # In that case, need to add all parts
    if comp_role == "page":
        for x in div.findAll("div"):
            div_parts = _get_part_dict(x, None)
    else:
        div_parts = _get_part_dict(div, comp_role)
    
    # for each illustration, store its coordinates and any potential caption
    if div.get("LABEL").lower() == 'illustration':
        image_parts[p] = {
                "legacy_parts": div_parts,
                "coords": div.find("area", {"SHAPE": "RECT"}).get("COORDS"),
            }
        # keep track of which illustration it is to make sure we can connect them back after
        last_img_part_id = p
        last_img_part_idx = idx
    # if the next element is a caption, attach it directly
    if div.get("LABEL").lower() == 'caption':
        if idx-1 == last_img_part_idx:
            image_parts[last_img_part_id]['caption_parts'] = div_parts
        else:
            msg = f"self.id, {div_parts['comp_page_no']} - caption {div.get('ID')} does not follow an illustration!"
    
    parts.append(div_parts)

parts

[{'comp_role': 'pagearea',
  'comp_id': 'pa0001001',
  'comp_label': 'textblock',
  'comp_fileid': 'img0001-alto',
  'comp_page_no': 1},
 {'comp_role': 'pagearea',
  'comp_id': 'pa0001002',
  'comp_label': 'textblock',
  'comp_fileid': 'img0001-alto',
  'comp_page_no': 1},
 {'comp_role': 'pagearea',
  'comp_id': 'pa0001003',
  'comp_label': 'textblock',
  'comp_fileid': 'img0001-alto',
  'comp_page_no': 1},
 {'comp_role': 'pagearea',
  'comp_id': 'pa0001004',
  'comp_label': 'textblock',
  'comp_fileid': 'img0001-alto',
  'comp_page_no': 1},
 {'comp_role': 'pagearea',
  'comp_id': 'pa0001005',
  'comp_label': 'illustration',
  'comp_fileid': 'img0001-alto',
  'comp_page_no': 1},
 {'comp_role': 'pagearea',
  'comp_id': 'pa0001006',
  'comp_label': 'caption',
  'comp_fileid': 'img0001-alto',
  'comp_page_no': 1},
 {'comp_role': 'pagearea',
  'comp_id': 'pa0001007',
  'comp_label': 'illustration',
  'comp_fileid': 'img0001-alto',
  'comp_page_no': 1}]

In [55]:
image_parts

{'pa0001005': {'legacy_parts': {'comp_role': 'pagearea',
   'comp_id': 'pa0001005',
   'comp_label': 'illustration',
   'comp_fileid': 'img0001-alto',
   'comp_page_no': 1},
  'coords': '298,4676,2539,5835',
  'caption_parts': {'comp_role': 'pagearea',
   'comp_id': 'pa0001006',
   'comp_label': 'caption',
   'comp_fileid': 'img0001-alto',
   'comp_page_no': 1}},
 'pa0001007': {'legacy_parts': {'comp_role': 'pagearea',
   'comp_id': 'pa0001007',
   'comp_label': 'illustration',
   'comp_fileid': 'img0001-alto',
   'comp_page_no': 1},
  'coords': '1649,4794,2147,4985'}}

In [53]:
content_item = {
    "m": {
        "id": f"{bl_issue.id}-i{str(counter).zfill(4)}",
        "tp": div_type,
        "pp": [],
    },
    "l": {
        "bl_nlp": bl_issue.nlp,
        "id": item_div.get("ID"),
        "parts": parts,
    },
}
for p in content_item["l"]["parts"]:
    pge_no = p["comp_page_no"]
    if pge_no not in content_item["m"]["pp"]:
        content_item["m"]["pp"].append(pge_no)

content_item

{'m': {'id': 'ILOL-1843-05-21-a-i0001', 'tp': 'article', 'pp': [1]},
 'l': {'bl_nlp': '0003005',
  'id': 'art0001',
  'parts': [{'comp_role': 'pagearea',
    'comp_id': 'pa0001001',
    'comp_label': 'textblock',
    'comp_fileid': 'img0001-alto',
    'comp_page_no': 1},
   {'comp_role': 'pagearea',
    'comp_id': 'pa0001002',
    'comp_label': 'textblock',
    'comp_fileid': 'img0001-alto',
    'comp_page_no': 1},
   {'comp_role': 'pagearea',
    'comp_id': 'pa0001003',
    'comp_label': 'textblock',
    'comp_fileid': 'img0001-alto',
    'comp_page_no': 1},
   {'comp_role': 'pagearea',
    'comp_id': 'pa0001004',
    'comp_label': 'textblock',
    'comp_fileid': 'img0001-alto',
    'comp_page_no': 1},
   {'comp_role': 'pagearea',
    'comp_id': 'pa0001005',
    'comp_label': 'illustration',
    'comp_fileid': 'img0001-alto',
    'comp_page_no': 1},
   {'comp_role': 'pagearea',
    'comp_id': 'pa0001006',
    'comp_label': 'caption',
    'comp_fileid': 'img0001-alto',
    'comp_page_n

### 5. BlOmniNewspaperPage Class

#### Debug page

In [11]:
[page.number for page in bl_issue.pages]

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]

In [18]:
image_cis = [i for i in bl_issue.issue_data['i'] if i['m']['tp']=='image']
image_cis

[{'m': {'id': 'ILOL-1843-05-28-a-i0002',
   'tp': 'image',
   'pp': [1],
   'iiif_link': 'https://impresso-project.ch/api/proxy/iiif/ILOL-1843-05-28-a-p0001/info.json',
   'var_t': 'Illustrated London Life',
   'ro': 2},
  'l': {'bl_nlp': '0003005',
   'src_files': {'mets_xml': '0003005_18430528_mets.xml',
    'alto_xml': ['0003005_18430528_0001.xml'],
    'page_image': ['0003005_18430528_0001.jp2']},
   'id': 'pa0001005',
   'parts': [{'comp_role': 'pagearea',
     'comp_id': 'pa0001005',
     'comp_label': 'illustration',
     'comp_fileid': 'img0001-alto',
     'comp_page_no': 1,
     'coords': [205, 61, 4154, 2270]}]},
  'c': [205, 61, 4154, 2270],
  'pOf': 'ILOL-1843-05-28-a-i0001'},
 {'m': {'id': 'ILOL-1843-05-28-a-i0003',
   'tp': 'image',
   'pp': [1],
   'iiif_link': 'https://impresso-project.ch/api/proxy/iiif/ILOL-1843-05-28-a-p0001/info.json',
   'var_t': 'Illustrated London Life',
   'ro': 3},
  'l': {'bl_nlp': '0003005',
   'src_files': {'mets_xml': '0003005_18430528_mets.

In [ ]:
ci_39 = [i for i in bl_issue.issue_data['i'] if i['m']['id']=='ILOL-1843-05-21-a-i0039']
ci_39

In [15]:
temp = [({'c': [1813, 2301, 515, 41], 't': [{'c': [1813, 2301, 216, 41], 'tx': 'LONDON,'}, {'c': [2050, 2306, 107, 33], 'tx': 'MAY'}, {'c': [2178, 2307, 52, 35], 'tx': '28,'}, {'c': [2254, 2307, 74, 31], 'tx': '1843.'}]}, [])]
lines, new_notes =list(zip(*temp))
new_notes = [i for n in new_notes for i in n]
new_notes

[]

In [9]:
for p in bl_issue.pages:
    p.add_issue(bl_issue)
    p.parse()

#bl_issue.pages[0].page_data

In [10]:
bl_issue.pages[0].page_data['r'][0]['p'][0]['l']

[{'c': [1813, 2301, 515, 41],
  't': [{'c': [1813, 2301, 216, 41], 'tx': 'LONDON,'},
   {'c': [2050, 2306, 107, 33], 'tx': 'MAY'},
   {'c': [2178, 2307, 52, 35], 'tx': '28,'},
   {'c': [2254, 2307, 74, 31], 'tx': '1843.'}]}]

In [13]:
page_1_xml = bl_issue.pages[0].xml

pg_1_printspace = page_1_xml.find("PrintSpace")
pg_1_printspace

<PrintSpace HEIGHT="8657" HPOS="224" ID="P1_PS00001" VPOS="0" WIDTH="5620">
<ComposedBlock HEIGHT="2231" HPOS="169" ID="P1_CB00001" TYPE="Illustration" VPOS="0" WIDTH="4214">
<Shape>
<Polygon POINTS="169,2175 183,1880 211,1847 224,1843 246,1843 271,1854 306,1813 315,1802 313,1794 328,1794 381,1792 392,1792 405,1728 365,1592 365,1589 479,902 479,901 909,580 915,576 2732,94 3808,0 4383,0 4383,668 3836,2154 3819,2154 3668,2118 3608,2170 3594,2178 3587,2181 3580,2183 3567,2183 2047,1965 2046,1964 2027,1943 1909,1928 1780,1877 1718,1876 1712,1901 1710,1901 1690,1902 1652,1906 1617,1920 1608,1935 1609,1953 1746,1972 1762,1992 1762,1994 1110,2069 1101,2069 1022,2069 917,2061 882,2031 849,2061 474,2165 457,2169 409,2172 406,2172 397,2165 391,2162 381,2153 345,2109 235,2088 228,2156 198,2231 169,2231"/>
</Shape>
<TextBlock HEIGHT="2231" HPOS="169" ID="P1_TB00001" STYLEREFS="TXT_0 PAR_LEFT" VPOS="0" WIDTH="4214">
<Shape>
<Polygon POINTS="169,2175 183,1880 211,1847 224,1843 246,1843 271,1854 306,

In [ ]:
mets_doc = bl_issue.xml
content_items = []

# Get logical structure of issue
divs = (
    mets_doc.find("structMap", {"TYPE": "LOGICAL"})
    .find("div", {"TYPE": "ISSUE"})
    .findChildren("div")
)

found_types = set(x.get("TYPE") for x in divs)

phys_structmap = mets_doc.find("structMap", {"TYPE": "PHYSICAL"})
structlink = mets_doc.find("structLink")

In [15]:
structlink

<mets:structLink>
<mets:smLinkGrp>
<mets:smLocatorLink xlink:href="#log1" xlink:label="issue" xlink:type="locator"/>
<mets:smLocatorLink xlink:href="#phys0" xlink:label="physSeq" xlink:type="locator"/>
<mets:smArcLink ARCTYPE="logicalphysical" xlink:from="issue" xlink:to="physSeq" xlink:type="arc"/>
</mets:smLinkGrp>
<mets:smLinkGrp>
<mets:smLocatorLink xlink:href="#art0001" xlink:label="article" xlink:type="locator"/>
<mets:smLocatorLink xlink:href="#pa0001001" xlink:label="page1 area1" xlink:type="locator"/>
<mets:smLocatorLink xlink:href="#pa0001002" xlink:label="page1 area2" xlink:type="locator"/>
<mets:smLocatorLink xlink:href="#pa0001003" xlink:label="page1 area3" xlink:type="locator"/>
<mets:smLocatorLink xlink:href="#pa0001004" xlink:label="page1 area4" xlink:type="locator"/>
<mets:smLocatorLink xlink:href="#pa0001005" xlink:label="page1 area5" xlink:type="locator"/>
<mets:smLocatorLink xlink:href="#pa0001006" xlink:label="page1 area6" xlink:type="locator"/>
<mets:smLocatorLink

In [25]:
mets_doc = bl_issue.xml
structlink = mets_doc.find("structLink")

all_linked_regions = [e.get('xlink:href').lstrip('#') for e in structlink.find_all('smLocatorLink')]
all_linked_regions

['log1',
 'phys0',
 'art0001',
 'pa0001001',
 'pa0001002',
 'pa0001003',
 'pa0001004',
 'pa0001005',
 'pa0001006',
 'pa0001007',
 'art0002',
 'pa0001008',
 'pa0001009',
 'pa0001010',
 'art0003',
 'pa0001011',
 'pa0001012',
 'pa0001013',
 'art0004',
 'pa0002001',
 'pa0002002',
 'pa0002003',
 'pa0002004',
 'pa0002005',
 'pa0002006',
 'pa0002007',
 'pa0002008',
 'art0005',
 'pa0002009',
 'pa0002010',
 'pa0002011',
 'pa0002012',
 'pa0002013',
 'art0006',
 'pa0002014',
 'pa0002015',
 'art0007',
 'pa0002016',
 'pa0002017',
 'pa0002018',
 'art0008',
 'pa0002019',
 'pa0002020',
 'art0009',
 'pa0002021',
 'pa0002022',
 'art0010',
 'pa0002023',
 'pa0002024',
 'pa0002025',
 'pa0002026',
 'art0011',
 'pa0002027',
 'art0012',
 'pa0002028',
 'pa0002029',
 'pa0002030',
 'pa0002031',
 'pa0002032',
 'pa0002033',
 'pa0002034',
 'pa0002035',
 'art0013',
 'pa0002036',
 'pa0002037',
 'pa0002038',
 'pa0002039',
 'pa0002040',
 'pa0002041',
 'pa0002042',
 'pa0002043',
 'pa0002044',
 'pa0003001',
 'pa0003002',

In [30]:
type(structlink)

bs4.element.Tag

In [4]:
IMG_COMP_LABELS = ["illustration", "image"]

def find_unlinked_image_cis(issue, structlink, ci_counter):
    # extract the list of all regions/blocks listed in the mets file
    all_linked_regions = [e.get('xlink:href').lstrip('#') for e in structlink.find_all('smLocatorLink')]
    image_cis = []

    for page in issue.pages:
        print(f"page {page.number}")
        pg_xml = page.xml
        pt_space = pg_xml.find("PrintSpace")

        for block in pt_space.children:
            if isinstance(block, bs4.element.NavigableString):
                continue
            
            # if the block is an illustration which was not attached to an existing CI, create a CI for it.
            if block.get("TYPE") and block.get("TYPE").lower() in IMG_COMP_LABELS and block.get("ID") not in all_linked_regions:
                coords = distill_coordinates(block)

                content_item = {
                    "m": {
                        #"id": f"{self.id}-i{str(counter).zfill(4)}",
                        "id": f"{issue.id}-i{str(ci_counter).zfill(4)}",
                        "tp": CONTENTITEM_TYPE_IMAGE,
                        "pp": [page.number],
                        "iiif_link": os.path.join(
                            IIIF_ENDPOINT_URI, f"{issue.id}-p{str(page.number).zfill(4)}", IIIF_SUFFIX
                        ),
                        "var_t": issue.var_title,
                    },
                    "l": {
                        "bl_nlp": issue.nlp,
                        "src_files": {
                            "mets_xml": os.path.basename(issue.mets_file),
                            "alto_xml": [
                                os.path.basename(issue.mets_file).replace(
                                    "mets", str(page.number).zfill(4)
                                )
                            ],
                            "page_image": [issue.page_filenames[page.number]],
                        },
                        "id": block.get("ID"),
                        "parts": [{
                            'comp_id': block.get("ID"),
                            'comp_label': block.get("TYPE").lower(),
                            'comp_fileid': f'img{str(page.number).zfill(3)}-alto',
                            'comp_page_no': page.number
                        }],
                    },
                    "c": coords,
                }
                print(f"page {page.number} -> found an unlinked illustration: {block.get('ID')}, coords = {coords}, adding the CI: {issue.id}-i{str(ci_counter).zfill(4)}")

                image_cis.append(content_item)
                ci_counter += 1

    return image_cis

In [28]:
find_unlinked_image_cis(bl_issue, structlink, len(bl_issue.issue_data['i'])+1)

page 1
page 1 -> found an unlinked illustration: P1_CB00001, coords = [169, 0, 4214, 2231], adding the CI: ILOL-1843-05-21-a-i0073
page 1 -> found an unlinked illustration: P1_CB00002, coords = [213, 1669, 187, 170], adding the CI: ILOL-1843-05-21-a-i0074
page 1 -> found an unlinked illustration: P1_CB00003, coords = [1609, 1868, 512, 153], adding the CI: ILOL-1843-05-21-a-i0075
page 1 -> found an unlinked illustration: P1_CB00004, coords = [707, 1978, 202, 224], adding the CI: ILOL-1843-05-21-a-i0076
page 1 -> found an unlinked illustration: P1_CB00005, coords = [3688, 2143, 159, 146], adding the CI: ILOL-1843-05-21-a-i0077
page 1 -> found an unlinked illustration: P1_CB00006, coords = [3587, 2126, 140, 169], adding the CI: ILOL-1843-05-21-a-i0078
page 1 -> found an unlinked illustration: P1_CB00007, coords = [545, 2409, 1709, 1128], adding the CI: ILOL-1843-05-21-a-i0079
page 2
page 3
page 4
page 5
page 6
page 7
page 8
page 9
page 10
page 11
page 12


[{'m': {'id': 'ILOL-1843-05-21-a-i0073',
   'tp': 'image',
   'pp': [1],
   'iiif_link': 'https://impresso-project.ch/api/proxy/iiif/ILOL-1843-05-21-a-p0001/info.json',
   'var_t': 'Illustrated London Life'},
  'l': {'bl_nlp': '0003005',
   'src_files': {'mets_xml': '0003005_18430521_mets.xml',
    'alto_xml': ['0003005_18430521_0001.xml'],
    'page_image': ['0003005_18430521_0001.jp2']},
   'id': 'P1_CB00001',
   'parts': [{'comp_id': 'P1_CB00001',
     'comp_label': 'illustration',
     'comp_fileid': 'img001-alto',
     'comp_page_no': 1}]},
  'c': [169, 0, 4214, 2231]},
 {'m': {'id': 'ILOL-1843-05-21-a-i0074',
   'tp': 'image',
   'pp': [1],
   'iiif_link': 'https://impresso-project.ch/api/proxy/iiif/ILOL-1843-05-21-a-p0001/info.json',
   'var_t': 'Illustrated London Life'},
  'l': {'bl_nlp': '0003005',
   'src_files': {'mets_xml': '0003005_18430521_mets.xml',
    'alto_xml': ['0003005_18430521_0001.xml'],
    'page_image': ['0003005_18430521_0001.jp2']},
   'id': 'P1_CB00002',
  

In [ ]:
IMG_COMP_LABELS = ["illustration", "image"]

non_linked_imgs = {}

for page in bl_issue.pages:
    print(f"page {page.number}")
    pg_xml = page.xml
    pt_space = pg_xml.find("PrintSpace")

    for block in pt_space.children:
        if isinstance(block, bs4.element.NavigableString):
            continue
        if block.get("TYPE") and block.get("TYPE").lower() in IMG_COMP_LABELS:
            if block.get("ID") not in all_linked_regions:
                coords = distill_coordinates(block)
                print(f"-> Unlinked illustration: {block.get('ID')}, coords = {coords}")
                if page.id not in non_linked_imgs:
                    non_linked_imgs[page.id] = {block.get('ID'): coords}
                else:
                    non_linked_imgs[page.id][block.get('ID')] = coords
            else:
                print(f"Linked illustration: {block.get('ID')}, coords = {distill_coordinates(block)}")

page 1
-> Unlinked illustration: P1_CB00001, coords = [169, 0, 4214, 2231]
-> Unlinked illustration: P1_CB00002, coords = [213, 1669, 187, 170]
-> Unlinked illustration: P1_CB00003, coords = [1609, 1868, 512, 153]
-> Unlinked illustration: P1_CB00004, coords = [707, 1978, 202, 224]
-> Unlinked illustration: P1_CB00005, coords = [3688, 2143, 159, 146]
-> Unlinked illustration: P1_CB00006, coords = [3587, 2126, 140, 169]
-> Unlinked illustration: P1_CB00007, coords = [545, 2409, 1709, 1128]
Linked illustration: pa0001005, coords = [298, 4676, 2241, 1159]
Linked illustration: pa0001007, coords = [1649, 4794, 498, 191]
page 2
page 3
Linked illustration: pa0003010, coords = [186, 457, 2499, 2970]
page 4
page 5
page 6
Linked illustration: pa0006012, coords = [669, 906, 714, 1006]
Linked illustration: pa0006013, coords = [617, 3009, 803, 1084]
Linked illustration: pa0006014, coords = [646, 4874, 748, 1072]
Linked illustration: pa0006015, coords = [1942, 1035, 690, 1008]
Linked illustration: p

In [11]:
pg_1_ci_regions = {}

for region in bl_issue.pages[0].page_data['r']:
    if 'pOf' in region:
        og_ci_id = region['pOf']
    else:
        og_ci_id = "No attached CI"
    if og_ci_id in pg_1_ci_regions:
        pg_1_ci_regions[og_ci_id].append(region)
    else:
        pg_1_ci_regions[og_ci_id] = [region]

pg_1_ci_regions.keys()

dict_keys(['No attached CI', 'ILOL-1843-05-21-a-i0001', 'ILOL-1843-05-21-a-i0002', 'ILOL-1843-05-21-a-i0004', 'ILOL-1843-05-21-a-i0005'])

In [5]:
def find_regions_per_ci(page_regions):
    ci_regions = {}
    unattached_counter = 1
    for region in page_regions:

        if 'pOf' in region:
            og_ci_id = region['pOf']
        else:
            og_ci_id = f"No attached CI {unattached_counter}"
            unattached_counter += 1

        if og_ci_id in ci_regions:
            ci_regions[og_ci_id].append(region)
        else:
            ci_regions[og_ci_id] = [region]
    return ci_regions

##### Printing the content items and their boxes on the pages

In [6]:
def draw_box_on_img(
    base_img_path: str, coords_xy: list, img: Image = None, width: int = 10, color="red", text=None
) -> Image:
    """Draw a bounding box on an image given coordinates in x1y1x2y2 format.

    The image can either be provided through its path, or as a PIL.Image object (specifically
    if other bboxes have already been drawn on it.)

    Args:
        base_img_path (str): Path to the image to open as a PIL Image.
        coords_xy (list): Coordinates of the bbox to draw.
        img (Image, optional): PIL image if already loaded. Defaults to None.
        width (int, optional): Stroke width for the bbox. Defaults to 10.
        color (str, optional): Color of stroke. Defaults to 'red'.

    Returns:
        Image: Resulting PIL Image with the bbox drawn on it.
    """
    if not img:
        img = Image.open(base_img_path)
        img = img.convert('RGB')
        img = img.convert(mode='1')
        img = img.convert('RGB')

    img_draw = ImageDraw.Draw(img)
    img_draw.rectangle(coords_xy, outline=color, width=width)

    if text is not None:
        #font=ImageFont.truetype("arial.ttf",10 ) 
        img_draw.text((coords_xy[0]+50, coords_xy[1]+75), text, 'black')#,font=font)
    return img

In [28]:
ci_regions = find_regions_per_ci(bl_issue.pages[0].page_data['r'])

In [7]:
colors = ["red", 'green', 'blue', 'purple', 'orange', 'cyan', 'brown', "limegreen", 'pink']

def print_regions_on_page(page_object, issue, page_info, img_blocks=None, color_to_print='all', save=False):
    print(f"Page: {page_object.id}")
    
    img_dir_path = os.path.join(page_info['img_dir_path'], page_info['new_filename'])
    
    img = Image.open(img_dir_path)
    img = img.convert('RGB')
    #img = img.convert(mode='1')
    #img = img.convert('RGB')
    if img_blocks:
        for bk_num, (block_id, block_coords) in enumerate(img_blocks.items()):
            bk_color = colors[bk_num%len(colors)]
            print(f"{block_id}-reg{bk_num} --> {bk_color}, coords = {block_coords}")
            img = draw_box_on_img(img_dir_path, coords_to_xy(block_coords), img, color=bk_color, text=f"{block_id}-reg{bk_num}")
    else:
        ci_regions = find_regions_per_ci(page_object.page_data['r'])

        other_cis_on_page = [ci for ci in issue.issue_data['i'] if page_object.number in ci['m']['pp'] and ci['m']['tp'] == 'image']
        all_img_coords = []
        for ci_num, ci in enumerate(other_cis_on_page):
                
            img_coords = coords_to_xy(ci['c'])
            all_img_coords.append(img_coords)
            #img = draw_box_on_img(img_dir_path, region_coords, img, text=f"{ci}-reg{r_num}")
            if 'pOf' in ci:
                reg_color = colors[int(ci['pOf'][-4:])%len(colors)]
                print(f"{ci['m']['id']}-img_pOf {ci['pOf']} --> CI {reg_color}, coords= {ci['c']}")
            else:
                reg_color = colors[ci_num%len(colors)]
                print(f"{ci['m']['id']} --> additional CI {reg_color}, coords= {ci['c']}")
            if color_to_print=='all' or reg_color == color_to_print:
                #print(region['c'])
                img = draw_box_on_img(img_dir_path, img_coords, img, color=reg_color, text=f"{ci['m']['id']}-no_reg")

        
        for ci_num, (ci, ci_regions) in enumerate(ci_regions.items()):
            for r_num, region in enumerate(ci_regions):
                if region['c'] not in all_img_coords:
                    # don't print twice the image blocks not attached to issues
                    region_coords = coords_to_xy(region['c'])
                    #img = draw_box_on_img(img_dir_path, region_coords, img, text=f"{ci}-reg{r_num}")
                    if "No attached" in ci:
                        reg_color = colors[ci_num%len(colors)]
                    else:
                        reg_color = colors[int(ci[-4:])%len(colors)]
                    print(f"{ci}-reg{r_num} --> {reg_color}, coords= {region['c']}")
                    if color_to_print=='all' or reg_color == color_to_print:
                        #print(region['c'])
                        img = draw_box_on_img(img_dir_path, region_coords, img, color=reg_color, text=f"{ci}-reg{r_num}")
                    #t_coords =  (region_coords[0], region_coords[1] + 100)
                    #img.text(t_coords, f"{ci}-reg{r_num}", fill=color_id, font_size=100)

        
    print(f"page {page_object.id} regions and CIs")
    img.show()
    if save:
        img.save(os.path.join(bl_sample_dir, "importer_examples", issue.id, f"{page_object.id}.jpg"))

In [ ]:
for page_object in bl_issue.pages:
    if page_object.id in non_linked_imgs:
        print_regions_on_page(page_object, non_linked_imgs[page_object.id])
        #break

In [32]:
for page_object in bl_issue.pages:
    if page_object.number in [1, 3, 6, 10]:
        # print the pages with the most images
        print_regions_on_page(page_object, bl_issue)#, color_to_print='purple')
    

### Second test issue:

In [ ]:
other_test_issue = selected[10]
other_test_issue

with open(os.path.join(test_issue.path, RENAMING_INFO_FILE), 'r') as fin:
    test_issue_renaming_info = json.load(fin)

test_issue_renaming_info

## Visualize the generated canonical data with the bbox visualizer

In [4]:
from impresso_essentials.bbox_visualizer.json_builder import build_bbox_json
from impresso_essentials.io.s3 import list_canonical_files
from text_preparation.importers.mets_alto.alto import parse_textline
from text_preparation.tokenization import insert_whitespace


#json_aubo = build_bbox_json("AUBO-1821-05-27-a")
#json_aubo

BL_aliases_to_check = ['ALBN', 'ANTWT', 'AUBO', "BHFA", "BRLU"]

config_sanity_ceck = {
    "titles": {
        "ALBN": ["1853/03/09"],
        "ANWT": ["1886/04/10"],
        "AUBO": ["1821/11/11"],
        "BHFA": ["1890/04/08"],
        "BRMG": ["1865/03/25"],
        "BRAA": ["1843/01/14"],
        "ILOL": ["1843/05/21", "1843/07/16"],
    },
    "exclude_titles": [],
    "year_only": False
}

selected = select_issues(bl_source_data_dir, config = config_sanity_ceck)

print(f"Selected  {len(selected)} issues in total")

for title in config_sanity_ceck['titles']:
    print(f"Detected {len([i for i in selected if i.alias == title])} issues for {title}")

Selected  7 issues in total
Detected 1 issues for ALBN
Detected 1 issues for ANWT
Detected 1 issues for AUBO
Detected 1 issues for BHFA
Detected 1 issues for BRMG
Detected 0 issues for BRAA
Detected 2 issues for ILOL


In [5]:
sc_issues = [BlOmniNewspaperIssue(i) for i in selected]

sc_pages = {}
for i in sc_issues:
    sc_pages[i] = []
    for p in i.pages:
        p.add_issue(i)
        p.parse()
        sc_pages[i].append(p)

sc_issues, sc_pages

<TextLine HEIGHT="44" HPOS="748" ID="P4_TL00022" VPOS="1243" WIDTH="436">
<String CC="0000000" CONTENT="SUNDAY," HEIGHT="44" HPOS="748" ID="word000070" VPOS="1243" WC="1.00" WIDTH="228"/>
<SP HPOS="976" ID="P4_SP00060" VPOS="1287" WIDTH="128"/>
<String CC="0000" CONTENT="Nov." HEIGHT="38" HPOS="1000" ID="word000071" VPOS="1243" WC="1.00" WIDTH="104"/>
<SP HPOS="1104" ID="P4_SP00061" VPOS="1287" WIDTH="80"/>
<String CC="000" CONTENT="11." HEIGHT="35" HPOS="1126" ID="word000072" VPOS="1243" WC="1.00" WIDTH="58"/>
<SP HPOS="0" ID="P4_SP00062" VPOS="1287" WIDTH="748"/>
</TextLine>
<TextLine HEIGHT="40" HPOS="820" ID="P1_TL00357" VPOS="5899" WIDTH="1244">
<String CC="090" CONTENT="THE" HEIGHT="22" HPOS="820" ID="word003343" VPOS="5915" WC="0.67" WIDTH="69"/>
<SP HPOS="890" ID="P1_SP02239" VPOS="5937" WIDTH="172"/>
<String CC="0000000" CONTENT="COTTAGE" HEIGHT="30" HPOS="910" ID="word003344" VPOS="5909" WC="1.00" WIDTH="151"/>
<SP HPOS="1061" ID="P1_SP02240" VPOS="5939" WIDTH="61"/>
<String 

([<text_preparation.importers.bl.omni.classes.BlOmniNewspaperIssue at 0x7f6ed01132d0>,
 {<text_preparation.importers.bl.omni.classes.BlOmniNewspaperIssue at 0x7f6ed01132d0>: [<text_preparation.importers.bl.omni.classes.BlOmniNewspaperPage at 0x7f6e0409df90>,
  <text_preparation.importers.bl.omni.classes.BlOmniNewspaperIssue at 0x7f6df8b45d90>: [<text_preparation.importers.bl.omni.classes.BlOmniNewspaperPage at 0x7f6dfa754410>,
  <text_preparation.importers.bl.omni.classes.BlOmniNewspaperIssue at 0x7f6df8bdbf10>: [<text_preparation.importers.bl.omni.classes.BlOmniNewspaperPage at 0x7f6df95faed0>,
  <text_preparation.importers.bl.omni.classes.BlOmniNewspaperIssue at 0x7f6db7fa0190>: [<text_preparation.importers.bl.omni.classes.BlOmniNewspaperPage at 0x7f6db7fb3550>,
  <text_preparation.importers.bl.omni.classes.BlOmniNewspaperIssue at 0x7f6db7fc4d90>: [<text_preparation.importers.bl.omni.classes.BlOmniNewspaperPage at 0x7f6db1c4b790>,
  <text_preparation.importers.bl.omni.classes.BlOmniN

In [10]:
sc_issues_renaming_info = {}

for i in sc_issues:
    with open(os.path.join(i.path, RENAMING_INFO_FILE), 'r') as fin:
        sc_issues_renaming_info[i] = json.load(fin)

In [11]:
def print_canonical_sanity_check(sc_issues, sc_issues_renaming_info, issue_idx, page_nums=None):
    
    sc_issue = sc_issues[issue_idx]
    issue_renaming_info = sc_issues_renaming_info[sc_issue]
    for page_object in sc_issues[issue_idx].pages:
        if not page_nums or page_object.number in page_nums:
            # print the pages with the most images
            page_info = issue_renaming_info[str(page_object.number)]
            print_regions_on_page(page_object, sc_issues[issue_idx], page_info)#, color_to_print='purple')

##### Issue 0: `ALBN-1853-03-09`

In [8]:
sc_issues[5].issue_data['i']

[{'m': {'id': 'ILOL-1843-05-21-a-i0001',
   'tp': 'article',
   'pp': [1],
   'var_t': 'Illustrated London Life',
   'lg': 'en',
   'ro': 1},
  'l': {'bl_nlp': '0003005',
   'src_files': {'mets_xml': '0003005_18430521_mets.xml',
    'alto_xml': ['0003005_18430521_0001.xml'],
    'page_image': ['0003005_18430521_0001.jp2']},
   'id': 'art0001',
   'parts': [{'comp_role': 'pagearea',
     'comp_id': 'pa0001001',
     'comp_label': 'textblock',
     'comp_fileid': 'img0001-alto',
     'comp_page_no': 1},
    {'comp_role': 'pagearea',
     'comp_id': 'pa0001002',
     'comp_label': 'textblock',
     'comp_fileid': 'img0001-alto',
     'comp_page_no': 1},
    {'comp_role': 'pagearea',
     'comp_id': 'pa0001003',
     'comp_label': 'textblock',
     'comp_fileid': 'img0001-alto',
     'comp_page_no': 1},
    {'comp_role': 'pagearea',
     'comp_id': 'pa0001004',
     'comp_label': 'textblock',
     'comp_fileid': 'img0001-alto',
     'comp_page_no': 1},
    {'comp_role': 'pagearea',
     'c

In [24]:
headline_part = sc_issues[0].issue_data['i'][2]['l']['parts'][0]
headline_part

{'comp_role': 'pagearea',
 'comp_id': 'pa0001020',
 'comp_label': 'headline',
 'comp_fileid': 'img0001-alto',
 'comp_page_no': 1}

In [25]:
pg_ptspace = sc_issues[0].pages[0].xml.find('PrintSpace')

words = []
for block in pg_ptspace.find_all('TextBlock', {'ID':headline_part['comp_id']}):
    for line in block.find_all('TextLine'):
        print(line)
        words = [s.get('CONTENT') for s in line.find_all('String')]
#title_text = [parse_textline(tl) for tl in pg_ptspace.find('TextBlock', {'ID':headline_part['comp_id']}).find_all('TextLine')]
title_text = ""
lang = None
for i, token in enumerate(words):
    
    if i == 0 and i != len(words) - 1:
        insert_ws = insert_whitespace(token, words[i+1], None, lang)
        print(f"case 1: i={i}, token={token}, insert_ws={insert_ws}")
    elif len(words) == 1 or i == len(words) - 1:
        insert_ws = False
        print(f"case 2: i={i}, token={token}, insert_ws={insert_ws}")
    else:
        insert_ws = insert_whitespace(
            token, words[i+1], words[i-1], lang
        )
        print(f"case 4: i={i}, token={token}, insert_ws={insert_ws}")

    if insert_ws:

        title_text += f"{token} "
    else:
        title_text += token

words, title_text

<TextLine HEIGHT="61" HPOS="1132" ID="P1_TL00095" VPOS="1050" WIDTH="978">
<String CC="0000000000" CONTENT="WEDNESDAY," HEIGHT="60" HPOS="1132" ID="word000285" VPOS="1050" WC="1.00" WIDTH="439"/>
<SP HPOS="1572" ID="P1_SP00224" VPOS="1110" WIDTH="277"/>
<String CC="00000" CONTENT="MARCH" HEIGHT="49" HPOS="1613" ID="word000286" VPOS="1050" WC="1.00" WIDTH="236"/>
<SP HPOS="1849" ID="P1_SP00225" VPOS="1110" WIDTH="80"/>
<String CC="00" CONTENT="9," HEIGHT="57" HPOS="1890" ID="word000287" VPOS="1054" WC="1.00" WIDTH="39"/>
<SP HPOS="1929" ID="P1_SP00226" VPOS="1110" WIDTH="181"/>
<String CC="00000" CONTENT="1853." HEIGHT="46" HPOS="1973" ID="word000288" VPOS="1054" WC="1.00" WIDTH="137"/>
<SP HPOS="0" ID="P1_SP00227" VPOS="1110" WIDTH="1132"/>
</TextLine>
case 1: i=0, token=WEDNESDAY,, insert_ws=True
case 4: i=1, token=MARCH, insert_ws=True
case 4: i=2, token=9,, insert_ws=True
case 2: i=3, token=1853., insert_ws=False


(['WEDNESDAY,', 'MARCH', '9,', '1853.'], 'WEDNESDAY, MARCH 9, 1853.')

In [22]:
sc_issues[6].issue_data

{'id': 'ILOL-1843-07-16-a',
 'cdt': '2025-09-01 17:03:37',
 'ts': '2025-09-01T15:03:37Z',
 'st': 'newspaper',
 'sm': 'print',
 'olr': True,
 'i': [{'m': {'id': 'ILOL-1843-07-16-a-i0001',
    'tp': 'article',
    'pp': [1],
    'var_t': 'Illustrated London Life',
    't': 'LONDON, JULY 16, 1843',
    'lg': 'en',
    'ro': 1},
   'l': {'bl_nlp': '0003005',
    'src_files': {'mets_xml': '0003005_18430716_mets.xml',
     'alto_xml': ['0003005_18430716_0001.xml'],
     'page_image': ['0003005_18430716_0001.jp2']},
    'id': 'art0001',
    'parts': [{'comp_role': 'pagearea',
      'comp_id': 'pa0001001',
      'comp_label': 'headline',
      'comp_fileid': 'img0001-alto',
      'comp_page_no': 1},
     {'comp_role': 'pagearea',
      'comp_id': 'pa0001002',
      'comp_label': 'textblock',
      'comp_fileid': 'img0001-alto',
      'comp_page_no': 1},
     {'comp_role': 'pagearea',
      'comp_id': 'pa0001003',
      'comp_label': 'textblock',
      'comp_fileid': 'img0001-alto',
      'comp

In [21]:
sc_issues[6].pages[4].page_data

{'id': 'ILOL-1843-07-16-a-p0005',
 'cdt': '2025-09-01 17:03:29',
 'ts': '2025-09-01T15:03:29Z',
 'st': 'newspaper',
 'sm': 'print',
 'r': [{'c': [540, 524, 725, 35],
   'p': [{'c': [540, 524, 725, 35],
     'l': [{'c': [540, 524, 725, 35],
       't': [{'c': [540, 529, 91, 30], 'tx': 'THE'},
        {'c': [655, 528, 271, 31], 'tx': 'SHAKSPERE'},
        {'c': [948, 524, 317, 33], 'tx': 'AUTOGRAPHS.'}]}]}],
   'pOf': 'ILOL-1843-07-16-a-i0035'},
  {'c': [279, 606, 1243, 695],
   'p': [{'c': [279, 606, 1243, 695],
     'l': [{'c': [326, 606, 1191, 52],
       't': [{'c': [326, 617, 88, 41], 'tx': 'Very'},
        {'c': [439, 614, 208, 35], 'tx': 'remarkable'},
        {'c': [672, 616, 28, 30], 'tx': 'is'},
        {'c': [715, 616, 39, 39], 'tx': 'it,'},
        {'c': [780, 614, 77, 31], 'tx': 'that'},
        {'c': [879, 622, 36, 22], 'tx': 'so'},
        {'c': [939, 613, 63, 33], 'tx': 'few'},
        {'c': [1016, 611, 153, 44], 'tx': 'personal'},
        {'c': [1183, 613, 106, 28], 'tx'

In [ ]:
print_canonical_sanity_check(sc_issues, sc_issues_renaming_info, issue_idx=6, page_nums=[4, 5, 6])

### TEST image coords for BL-alias format

In [3]:
colors = ["red", 'green', 'blue', 'purple', 'orange', 'cyan', 'brown', "limegreen", 'pink']

def print_regions_on_page_from_jsons(issue_id, issue_json, page_json, issue_renaming_info, img_blocks=None, color_to_print='all', save=False):

    print(f"Page: {page_json['id']}")
    pg_num = int(page_json['id'][-4:])
    page_info = issue_renaming_info[str(pg_num)]
    img_dir_path = os.path.join(page_info['img_dir_path'], page_info['new_filename'])
    
    img = Image.open(img_dir_path)
    img = img.convert('RGB')
    #img = img.convert(mode='1')
    #img = img.convert('RGB')
    if img_blocks:
        for bk_num, (block_id, block_coords) in enumerate(img_blocks.items()):
            bk_color = colors[bk_num%len(colors)]
            print(f"{block_id}-reg{bk_num} --> {bk_color}, coords = {block_coords}")
            img = draw_box_on_img(img_dir_path, coords_to_xy(block_coords), img, color=bk_color, text=f"{block_id}-reg{bk_num}")
    else:
        ci_regions = find_regions_per_ci(page_json['r'])

        other_cis_on_page = [ci for ci in issue_json['i'] if pg_num in ci['m']['pp'] and ci['m']['tp'] == 'image']
        all_img_coords = []
        for ci_num, ci in enumerate(other_cis_on_page):
                
            img_coords = coords_to_xy(ci['c'])
            all_img_coords.append(img_coords)
            #img = draw_box_on_img(img_dir_path, region_coords, img, text=f"{ci}-reg{r_num}")
            if 'pOf' in ci:
                reg_color = colors[int(ci['pOf'][-4:])%len(colors)]
                print(f"{ci['m']['id']}-img_pOf {ci['pOf']} --> CI {reg_color}, coords= {ci['c']}")
            else:
                reg_color = colors[ci_num%len(colors)]
                print(f"{ci['m']['id']} --> additional CI {reg_color}, coords= {ci['c']}")
            if color_to_print=='all' or reg_color == color_to_print:
                #print(region['c'])
                img = draw_box_on_img(img_dir_path, img_coords, img, color=reg_color, text=f"{ci['m']['id']}-no_reg")

        
        for ci_num, (ci, ci_regions) in enumerate(ci_regions.items()):
            for r_num, region in enumerate(ci_regions):
                if region['c'] not in all_img_coords:
                    # don't print twice the image blocks not attached to issues
                    region_coords = coords_to_xy(region['c'])
                    #img = draw_box_on_img(img_dir_path, region_coords, img, text=f"{ci}-reg{r_num}")
                    if "No attached" in ci:
                        reg_color = colors[ci_num%len(colors)]
                    else:
                        reg_color = colors[int(ci[-4:])%len(colors)]
                    print(f"{ci}-reg{r_num} --> {reg_color}, coords= {region['c']}")
                    if color_to_print=='all' or reg_color == color_to_print:
                        #print(region['c'])
                        img = draw_box_on_img(img_dir_path, region_coords, img, color=reg_color, text=f"{ci}-reg{r_num}")
                    #t_coords =  (region_coords[0], region_coords[1] + 100)
                    #img.text(t_coords, f"{ci}-reg{r_num}", fill=color_id, font_size=100)

        
    print(f"page {page_json['id']} regions and CIs")
    img.show()
    if save:
        img.save(os.path.join(bl_sample_dir, "importer_examples", issue_id, f"{page_json['id']}.jpg"))

In [ ]:
img_path = "/mnt/impresso_ocr_BL_old/0000104/1881/0729/0000104_18810729_0001.jp2"

page_coords = coords_to_xy([1525,347,7808,8125])
summer_coords = [94,96,383,150]
the_coords = [800,90,929,130]
eagle_coords = [945,92,1136,135]
packing_coords = [1149,96,1377,141]
sentence_coords = [the_coords[0], the_coords[1], packing_coords[2], packing_coords[3]]
article_1_coords = [52,62,6335,7829]

img = draw_box_on_img(img_path, article_1_coords, width=20)

img

In [ ]:
img_pat_2 = "/mnt/impresso_ocr_BL_old/0000104/1881/0729/0000104_18810729_0002.jp2"

article_2_coords = [0,45,4945,7872]
article_3_coords = [4183,35,5642,7822]
article_4_coords = [4903,25,6350,7817]

img_2 = draw_box_on_img(img_pat_2, article_3_coords, width=20)
img_2 = draw_box_on_img(img_pat_2, article_4_coords, img_2, width=10)

img_2